In [ ]:
# Cell 1: Imports and configuration
import os
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

# ====== Config ======
device = torch.device("cpu")         # FORCED: CPU only (user requested no CUDA)
DATA_ROOT = "data"                   # root containing train/val/test/check as needed
TRAIN_ROOT = os.path.join(DATA_ROOT, "train")   # train folder with class subfolders
TEST_ROOT  = os.path.join(DATA_ROOT, "test")    # test folder for evaluation
IMAGE_SIZE = 1024                     # set to 1024 if you want high-res (slow)
N_WAY = 2                            # N-way classification per episode
K_SHOT = 5                           # K support examples per class
Q_QUERY = 5                          # Q query examples per class
EPISODES_PER_EPOCH = 100             # episodes used per epoch
EPOCHS = 20
LEARNING_RATE = 1e-3
SAVE_PATH = "fsl_encoder.pth"        # saved encoder weights
RANDOM_SEED = 42

# reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Configuration:")
print(f" device={device}, image_size={IMAGE_SIZE}, N={N_WAY}, K={K_SHOT}, Q={Q_QUERY}")
print(f" train_root={TRAIN_ROOT}, test_root={TEST_ROOT}")


Configuration:
 device=cpu, image_size=224, N=2, K=5, Q=5
 train_root=data\train, test_root=data\test


In [8]:
# Cell 2: Dataset helpers for episodic sampling (supports ImageFolder-like structure)

class SimpleFolder:
    """
    Lightweight loader that maps class -> list of image paths.
    Expects folder structure:
        root/
            classA/
                img1.jpg
                ...
            classB/
                ...
    """
    def __init__(self, root, exts=(".jpg", ".jpeg", ".png")):
        self.root = Path(root)
        self.class_to_paths = {}
        for cls_dir in sorted(self.root.iterdir()):
            if not cls_dir.is_dir():
                continue
            imgs = [str(p) for p in cls_dir.iterdir() if p.suffix.lower() in exts]
            if imgs:
                self.class_to_paths[cls_dir.name] = imgs
        self.classes = list(self.class_to_paths.keys())

    def get_classes(self):
        return self.classes

    def sample_support_query(self, selected_classes, k_shot, q_query):
        """
        For each class in selected_classes, sample k_shot support and q_query query images.
        Returns lists: support_paths, support_labels, query_paths, query_labels
        """
        support_paths, support_labels, query_paths, query_labels = [], [], [], []
        for i, cls in enumerate(selected_classes):
            pool = self.class_to_paths[cls]
            if len(pool) < k_shot + q_query:
                raise ValueError(f"Class {cls} does not have enough examples ({len(pool)}) for k+q={k_shot+q_query}")
            sampled = random.sample(pool, k_shot + q_query)
            supp = sampled[:k_shot]
            qry  = sampled[k_shot:]
            support_paths += supp
            support_labels += [i] * k_shot
            query_paths += qry
            query_labels += [i] * q_query
        return support_paths, support_labels, query_paths, query_labels


In [9]:
# Cell 3: Transforms and image loader utility (keeps memory small)
transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    # use ImageNet normalization (same as backbone expectation)
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_image_tensor(path):
    img = Image.open(path).convert("RGB")
    return transform(img)


In [10]:
# Cell 4: Prototypical network backbone (ResNet18 without final fc)
class ProtoNetEncoder(nn.Module):
    def __init__(self, backbone_name="resnet18", pretrained=False):
        super().__init__()
        if backbone_name == "resnet18":
            base = models.resnet18(weights=None)  # no pretrained
            # keep conv..pool layers, remove fc
            self.encoder = nn.Sequential(*list(base.children())[:-1])
            self.out_dim = base.fc.in_features
        else:
            raise ValueError("Only resnet18 implemented in this snippet")

    def forward(self, x):
        x = self.encoder(x)              # (B, C, 1, 1)
        x = x.view(x.size(0), -1)        # (B, out_dim)
        return x

# quick instantiate
encoder = ProtoNetEncoder().to(device)
print("Encoder output dim:", encoder.out_dim)


Encoder output dim: 512


In [11]:
# Cell 5: distance, loss, and one episode training step

def pairwise_euclidean(a, b):
    """
    a: (m, d)
    b: (n, d)
    returns distances (m, n) with squared euclidean distances
    """
    m, d = a.shape
    n = b.shape[0]
    # efficient computation: (a-b)^2 = a^2 + b^2 - 2ab
    aa = (a**2).sum(dim=1).unsqueeze(1)   # (m,1)
    bb = (b**2).sum(dim=1).unsqueeze(0)   # (1,n)
    ab = torch.matmul(a, b.t())           # (m,n)
    dists = aa + bb - 2*ab
    return dists

def prototypical_loss(prototypes, query_embeddings, query_labels):
    """
    prototypes: (N_way, d)
    query_embeddings: (N_way * Q, d)
    query_labels: (N_way * Q) values in [0..N_way-1]
    returns loss (scalar) and accuracy
    """
    dists = pairwise_euclidean(query_embeddings, prototypes)  # (num_queries, N_way)
    # logits as negative distances
    logits = -dists
    labels = query_labels.to(logits.device)
    loss = F.cross_entropy(logits, labels)
    preds = torch.argmax(logits, dim=1)
    acc = (preds == labels).float().mean().item()
    return loss, acc


In [12]:
# Cell 6: Episodic training loop
from tqdm import trange

# Prepare folder mapping
train_folder = SimpleFolder(TRAIN_ROOT)
classes = train_folder.get_classes()
if len(classes) < N_WAY:
    raise RuntimeError(f"Not enough classes in train folder ({len(classes)}) for N_WAY={N_WAY}")

optimizer = torch.optim.Adam(encoder.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS+1):
    encoder.train()
    epoch_loss = 0.0
    epoch_acc = 0.0

    for ep in range(EPISODES_PER_EPOCH):
        # sample N classes for this episode
        selected = random.sample(classes, N_WAY)
        support_paths, support_labels, query_paths, query_labels = train_folder.sample_support_query(selected, K_SHOT, Q_QUERY)

        # load support and query tensors
        support_tensors = torch.stack([load_image_tensor(p) for p in support_paths]).to(device)  # (N*K, C, H, W)
        query_tensors   = torch.stack([load_image_tensor(p) for p in query_paths]).to(device)    # (N*Q, C, H, W)
        support_labels_t = torch.tensor(support_labels, dtype=torch.long)
        query_labels_t   = torch.tensor(query_labels, dtype=torch.long)

        # forward
        s_emb = encoder(support_tensors)   # (N*K, d)
        q_emb = encoder(query_tensors)     # (N*Q, d)

        # compute prototypes (mean per class)
        d = s_emb.shape[1]
        prototypes = []
        for i in range(N_WAY):
            idxs = (support_labels_t == i).nonzero(as_tuple=True)[0]
            proto = s_emb[idxs].mean(dim=0)
            prototypes.append(proto)
        prototypes = torch.stack(prototypes)  # (N, d)

        loss, acc = prototypical_loss(prototypes, q_emb, query_labels_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc

    avg_loss = epoch_loss / EPISODES_PER_EPOCH
    avg_acc = epoch_acc / EPISODES_PER_EPOCH
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Episodic Acc: {avg_acc:.4f}")

# Save encoder
torch.save(encoder.state_dict(), SAVE_PATH)
print("Saved encoder to", SAVE_PATH)


Epoch 1/20 | Loss: 3.1348 | Episodic Acc: 0.7100
Epoch 2/20 | Loss: 0.3764 | Episodic Acc: 0.8310
Epoch 3/20 | Loss: 0.2810 | Episodic Acc: 0.8860
Epoch 4/20 | Loss: 0.1185 | Episodic Acc: 0.9550
Epoch 5/20 | Loss: 0.0909 | Episodic Acc: 0.9720
Epoch 6/20 | Loss: 0.0648 | Episodic Acc: 0.9750
Epoch 7/20 | Loss: 0.0362 | Episodic Acc: 0.9910
Epoch 8/20 | Loss: 0.0109 | Episodic Acc: 0.9960
Epoch 9/20 | Loss: 0.0031 | Episodic Acc: 0.9980
Epoch 10/20 | Loss: 0.0596 | Episodic Acc: 0.9810
Epoch 11/20 | Loss: 0.0390 | Episodic Acc: 0.9910
Epoch 12/20 | Loss: 0.0284 | Episodic Acc: 0.9910
Epoch 13/20 | Loss: 0.0141 | Episodic Acc: 0.9960
Epoch 14/20 | Loss: 0.0085 | Episodic Acc: 0.9970
Epoch 15/20 | Loss: 0.0032 | Episodic Acc: 0.9980
Epoch 16/20 | Loss: 0.0494 | Episodic Acc: 0.9830
Epoch 17/20 | Loss: 0.0583 | Episodic Acc: 0.9740
Epoch 18/20 | Loss: 0.0068 | Episodic Acc: 0.9980
Epoch 19/20 | Loss: 0.0005 | Episodic Acc: 1.0000
Epoch 20/20 | Loss: 0.0288 | Episodic Acc: 0.9910
Saved enc

In [ ]:
# -------------------------------------------------------
# Extract embeddings from the trained FSL encoder and save
# -------------------------------------------------------
import pickle
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# Define same preprocessing used during training
extract_transform = transforms.Compose([
    transforms.Resize((1024, 1024)),  # or your chosen resolution
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load dataset (the same as your training dataset)
dataset = datasets.ImageFolder(root=TRAIN_ROOT, transform=extract_transform)
loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2)

# Extract features and labels
encoder.eval()
all_features, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Extracting FSL features"):
        imgs = imgs.to(device)
        feats = encoder(imgs).cpu().numpy()
        all_features.append(feats)
        all_labels.append(labels.numpy())

X = np.concatenate(all_features)
y = np.concatenate(all_labels)

# Save as pickle for later evaluation
with open("fsl_features.pkl", "wb") as f:
    pickle.dump({"features": X, "labels": y}, f)

print("✅ Saved FSL features to fsl_features.pkl")


Extracting FSL features: 100%|██████████| 125/125 [00:22<00:00,  5.66it/s]

✅ Saved FSL features to fsl_features.pkl


In [14]:
# Cell 7: Evaluate on data/test using N-way K-shot episodes (report average accuracy)
def evaluate_few_shot(encoder, folder_root, n_way= N_WAY, k_shot=K_SHOT, q_query=Q_QUERY, episodes=200):
    folder = SimpleFolder(folder_root)
    classes = folder.get_classes()
    if len(classes) < n_way:
        raise RuntimeError(f"Not enough classes in {folder_root} for evaluation (need >= {n_way})")

    encoder.eval()
    accs = []
    for _ in range(episodes):
        selected = random.sample(classes, n_way)
        supp_paths, supp_labels, qry_paths, qry_labels = folder.sample_support_query(selected, k_shot, q_query)
        supp_t = torch.stack([load_image_tensor(p) for p in supp_paths]).to(device)
        qry_t  = torch.stack([load_image_tensor(p) for p in qry_paths]).to(device)
        supp_labels_t = torch.tensor(supp_labels, dtype=torch.long)
        qry_labels_t  = torch.tensor(qry_labels, dtype=torch.long)

        with torch.no_grad():
            s_emb = encoder(supp_t)
            q_emb = encoder(qry_t)

            prototypes = []
            for i in range(n_way):
                idxs = (supp_labels_t == i).nonzero(as_tuple=True)[0]
                proto = s_emb[idxs].mean(dim=0)
                prototypes.append(proto)
            prototypes = torch.stack(prototypes)

            _, acc = prototypical_loss(prototypes, q_emb, qry_labels_t)
            accs.append(acc)

    mean_acc = float(np.mean(accs))
    std_acc  = float(np.std(accs))
    print(f"Few-shot evaluation on {folder_root}: {mean_acc:.4f} ± {std_acc:.4f} (N={n_way}, K={k_shot}, Q={q_query})")
    return mean_acc, std_acc

# Run evaluation on test set
evaluate_few_shot(encoder, TEST_ROOT, n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY, episodes=200)


c:\Users\Dabie\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Few-shot evaluation on data\test: 0.9915 ± 0.0279 (N=2, K=5, Q=5)


(0.9914999979734421, 0.02788817618382644)

In [15]:
# Cell 8: Predict all images in data/check using prototypes built from a few shots per class.
# This is a simple deterministic predictor: build prototypes using K_SHOT support images
# from train set per class, then classify check images by nearest prototype.

def build_prototypes_from_train(encoder, train_root, k_shot_for_proto=K_SHOT):
    folder = SimpleFolder(train_root)
    prototypes = {}
    encoder.eval()
    for cls in folder.get_classes():
        paths = folder.class_to_paths[cls]
        if len(paths) < k_shot_for_proto:
            raise ValueError(f"Not enough examples in train class {cls} to build prototype")
        sampled = random.sample(paths, k_shot_for_proto)
        tensors = torch.stack([load_image_tensor(p) for p in sampled]).to(device)
        with torch.no_grad():
            emb = encoder(tensors)  # (k, d)
            proto = emb.mean(dim=0).cpu().numpy()
        prototypes[cls] = proto
    return prototypes

def predict_check_images(encoder, prototypes, check_root="data/check", out_txt="fsl_check_predictions.txt"):
    # check_root can be structured with subfolders (ImageFolder) or flat. We'll support subfolders.
    check_ds = SimpleFolder(check_root)
    classes_in_check = check_ds.get_classes()
    # We'll load all images across subfolders
    all_images = []
    for cls in classes_in_check:
        for p in check_ds.class_to_paths[cls]:
            all_images.append((p, cls))

    results = []
    encoder.eval()
    for img_path, true_cls in all_images:
        t = load_image_tensor(img_path).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = encoder(t).cpu().numpy()[0]
        # find nearest prototype (euclidean)
        best_cls, best_dist = None, float("inf")
        for cls_name, proto in prototypes.items():
            dist = np.linalg.norm(emb - proto)
            if dist < best_dist:
                best_dist = dist
                best_cls = cls_name
        results.append((img_path, true_cls, best_cls, best_dist))

        # show
        img = Image.open(img_path).convert("RGB")
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(f"{Path(img_path).name}\nTrue: {true_cls} Pred: {best_cls} (d={best_dist:.2f})")
        plt.axis("off")
        plt.show()

    # save results
    with open(out_txt, "w") as f:
        f.write("path,true,pred,dist\n")
        for p in results:
            f.write(f"{p[0]},{p[1]},{p[2]},{p[3]:.4f}\n")
    print("Saved predictions to", out_txt)
    return results

# Build prototypes and predict
protos = build_prototypes_from_train(encoder, TRAIN_ROOT, k_shot_for_proto=K_SHOT)
predict_check_images(encoder, protos, check_root=os.path.join(DATA_ROOT, "check"), out_txt="fsl_check_predictions.txt")


Saved predictions to fsl_check_predictions.txt


[]